## Silver layer

**Reads:** `permits.bronze.active_permits` and `permits.bronze.cleared_permits`
**Writes:** `permits.silver.permits`

Stacks both bronze tables into one, then cleans:

1. Trims whitespace on every text column and turns blank values into null
2. Builds `permit_key` from permit number + revision number + permit type, and keeps one row per key
3. Converts dates and numeric columns to real types
4. Cleans the cost column into `est_const_cost_clean` (junk text becomes null)
5. Flags secondary permits, duplicates of main permits, admin records, cancellations and impossible dates
6. Adds full address, days to issue, days to complete and application year

Writes with a **merge on `permit_key`**, so rerunning never creates duplicates.

Silver keeps every row and every original column. Filtering and renaming happen in gold.

In [0]:
from pyspark.sql import functions as F

from pyspark.sql.window import Window


spark.sql("CREATE SCHEMA IF NOT EXISTS permits.silver")



In [0]:
# spark.table is reading table from the bronze layer with adding a column named source_file so I know where is the row coming from.
active = spark.table("permits.bronze.active_permits").withColumn("source_table", F.lit("active"))

cleared = spark.table("permits.bronze.cleared_permits").withColumn("source_table", F.lit("cleared"))


df =  active.unionByName(cleared)

print(f"combine: {df.count():,} rows")



In [0]:
text_cols = [c for c in df.columns if c != "_loaded_at"]

for c in text_cols:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))
    )
print("trimmed", len(text_cols), "columns")


In [0]:
df = df.withColumn(
    "permit_key",
    F.concat_ws("|", F.col("PERMIT_NUM"), F.col("REVISION_NUM"), F.col("PERMIT_TYPE"))
)

# assuming even after creating a permit key that should create unique rows, if there are still duplicates those should be dropped since that is a data error.
total = df.count()
distinct_keys = df.select("permit_key").distinct().count()
print(f"duplicate rows to drop: {total - distinct_keys:,}")


# created a window function that orders by the latest application date so for next step we can keep the single latest record that have duplicates
w = Window.partitionBy("permit_key").orderBy(
    F.col("_loaded_at").desc(),
    F.col("APPLICATION_DATE").desc()
)

df = (
    df.withColumn("_row_num", F.row_number().over(w)).filter(F.col("_row_num") == 1))
df = df.drop("_row_num")
print(len(df.columns), "columns")



print(f"{df.count():,} rows")
print(f"{df.select('permit_key').distinct().count():,} distinct keys")


In [0]:
date_cols = ["APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE"]

for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))



In [0]:
for c in ["DWELLING_UNITS_CREATED", "DWELLING_UNITS_LOST", 
           "ASSEMBLY", "INSTITUTIONAL", "RESIDENTIAL",

            "BUSINESS_AND_PERSONAL_SERVICES", "MERCANTILE",
    "INDUSTRIAL", "INTERIOR_ALTERATIONS", "DEMOLITION"]:
    has_dot = df.filter(F.col(c).contains(".")).count()
    print(f"{c}: {has_dot:,} values with a decimal point")

In [0]:
df = df.withColumn(
    "est_const_cost_clean",
    F.when(
        F.col("EST_CONST_COST").rlike("^[0-9,.]+$"),
        F.regexp_replace(F.col("EST_CONST_COST"), ",", "").cast("double")
    ).otherwise(None)
)

In [0]:
numeric_cols = [
    "DWELLING_UNITS_CREATED", "DWELLING_UNITS_LOST","ASSEMBLY", "INSTITUTIONAL", "RESIDENTIAL",
    "BUSINESS_AND_PERSONAL_SERVICES", "MERCANTILE",
    "INDUSTRIAL", "INTERIOR_ALTERATIONS", "DEMOLITION",
]

for c in numeric_cols:
    df = df.withColumn(
        c, F.col(c).cast("double")

    )
for c in ["DWELLING_UNITS_CREATED", "RESIDENTIAL", "est_const_cost_clean"]:
    print(c, dict(df.dtypes)[c])

print(f"units created not null: {df.filter(F.col('DWELLING_UNITS_CREATED').isNotNull()).count():,}")


In [0]:
# Project number: strip the trailing suffix so BLD/PLB/HVA on the same job group together
df = df.withColumn("project_num", F.regexp_replace(F.col("PERMIT_NUM"), r"\s+\S+$", ""))

# Type-based: is this one of the duplicate-prone permit types?
df = df.withColumn("is_secondary_type", F.col("PERMIT_TYPE").rlike("(?i)plumbing|mechanical|drain|partial permit|conditional permit"))

# Data-based: does this project have a main permit anywhere in the data?
w_proj = Window.partitionBy("project_num")
df = df.withColumn("project_has_main", F.max(F.when(~F.col("is_secondary_type"), 1).otherwise(0)).over(w_proj) == 1)

# Exclude only if it's a secondary type AND a main permit exists for the same project.
# Standalone plumbing/drain jobs (60,753 of them) stay.
df = df.withColumn("is_duplicate_of_main", F.col("is_secondary_type") & F.col("project_has_main"))

# Rows that aren't building projects at all
df = df.withColumn("is_admin_record", F.col("PERMIT_TYPE").isin("DCs DeferredFees", "Toronto Buildings Contacts", "Building Historical data - Converted", "Toronto Building Standard Attachments", "Site Inspection(Scarborough)", "Change of Use Permit"))

df = df.withColumn("is_cancelled", F.col("STATUS").isin("Cancelled", "Closed - Dormant"))

df = df.withColumn("is_issued", F.col("ISSUED_DATE").isNotNull())

df = df.withColumn("has_bad_date", (F.col("COMPLETED_DATE") > F.current_date()) | (F.col("ISSUED_DATE") < F.col("APPLICATION_DATE")))

# Replaced by is_duplicate_of_main. Dropping it so gold can't filter on the wrong column.
df = df.drop("is_sub_permit")


print(f"is_secondary_type    {df.filter(F.col('is_secondary_type')).count():,}")
print(f"is_duplicate_of_main {df.filter(F.col('is_duplicate_of_main')).count():,}")
print(f"is_admin_record      {df.filter(F.col('is_admin_record')).count():,}")
print(f"is_sub_permit gone:  {'is_sub_permit' not in df.columns}")

In [0]:
df = df.withColumn("full_address", F.concat_ws(" ", F.col("STREET_NUM"), F.col("STREET_NAME"), F.col("STREET_TYPE"), F.col("STREET_DIRECTION")) )


df = df.withColumn("days_to_issue",
        F.datediff(F.col("ISSUED_DATE"), F.col("APPLICATION_DATE")))

df = df.withColumn("days_to_complete",
        F.datediff(F.col("COMPLETED_DATE"), F.col("ISSUED_DATE")))

df = df.withColumn("application_year", F.year(F.col("APPLICATION_DATE")))






In [0]:
from delta.tables import DeltaTable

target = "permits.silver.permits"

if not spark.catalog.tableExists(target):
    # First run: nothing to merge into, so just create the table.
    df.write.saveAsTable(target)
    print("created", target)
else:
    # Later runs: match on permit_key.
    # If the key exists, overwrite that row. If not, insert it.
    # This is what makes the job safe to re-run.
    (DeltaTable.forName(spark, target).alias("t")
        .merge(df.alias("s"), "t.permit_key = s.permit_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print("merged into", target)

print(f"final: {spark.table(target).count():,} rows")

In [0]:
s = spark.table(target)

# The rows that actually count: not duplicates of a main permit, not admin records
main = s.filter(~F.col("is_duplicate_of_main") & ~F.col("is_admin_record"))
main_count = main.count()

checks = {
    "total_rows":          s.count(),
    "distinct_keys":       s.select("permit_key").distinct().count(),
    "main_permits":        main_count,
    "secondary_type":      s.filter(F.col("is_secondary_type")).count(),
    "duplicate_of_main":   s.filter(F.col("is_duplicate_of_main")).count(),
    "standalone_secondary": s.filter(F.col("is_secondary_type") & ~F.col("is_duplicate_of_main")).count(),
    "admin_records":       s.filter(F.col("is_admin_record")).count(),
    "cost_usable_pct":     round(main.filter(F.col("est_const_cost_clean").isNotNull()).count() / main_count * 100, 1),
    "null_postal":         s.filter(F.col("POSTAL").isNull()).count(),
    "bad_dates":           s.filter(F.col("has_bad_date")).count(),
    "not_issued":          s.filter(~F.col("is_issued")).count(),
    "cancelled":           s.filter(F.col("is_cancelled")).count(),
}

for k, v in checks.items():
    print(f"{k:22} {v:,}" if isinstance(v, int) else f"{k:22} {v}")